# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a LangChain agent.
The agent picks its own tools, so you must **prove** the answer comes from tool output.

```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

## Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup | PROVIDED |
| 2 | Create creative tools | PROVIDED + YOUR TURN |
| 3 | Create the agent | YOUR TURN |
| 4 | Test with sample complaints | YOUR TURN |
| 5 | Analyze agent behavior | YOUR TURN |
| | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, tool evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langchain_integration.ipynb` (tools + `create_agent`)

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [3]:
Install required packages (uncomment to run)
pip install -U langchain langchain-openai python-dotenv

SyntaxError: invalid syntax (4193322444.py, line 1)

In [2]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon` [PROVIDED]

In [14]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

### 2b. `check_hawkins_records` [YOUR TURN]

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

In [3]:
@tool
def check_hawkins_records(query: str) -> str:
    """Search Hawkins records for a matching topic."""

    records = {
        "portal": "Hawkins reports say portals sometimes appear near the old laboratory.",
        "monsters": "Witnesses reported strange footprints near the forest, but no monster was confirmed.",
        "psychics": "Some residents said they sensed unusual activity, but their claims were not verified.",
        "electricity": "Residents reported flickering lights during unusual events. This does not prove a portal was nearby.",
    }

    query_lower = query.lower()

    for topic, record in records.items():
        if topic in query_lower:
            return record

    return f"No record found for: {query}"

In [5]:
print(check_hawkins_records.invoke({"query": "Tell me about the portal"}))

Hawkins reports say portals sometimes appear near the old laboratory.


### 2c. `cast_interdimensional_spell` [YOUR TURN]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

In [6]:
@tool
def cast_interdimensional_spell(
    problem: str,
    creativity_level: str = "medium"
) -> str:
    """Suggest creative spells for a problem."""

    levels = {
        "low": 1,
        "medium": 2,
        "high": 3,
    }

    number_of_spells = levels.get(creativity_level.lower(), 2)

    spells = [
        f"Build a magical fence to protect the garden from: {problem}.",
        f"Use colored lights to guide the danger away from: {problem}.",
        f"Ask the party to create a safe escape route for: {problem}.",
        f"Place a harmless decoy far away to distract attention from: {problem}.",
    ]

    selected_spells = random.sample(
        spells,
        k=min(number_of_spells, len(spells))
    )

    return "\n".join(selected_spells)

In [7]:
print(cast_interdimensional_spell.invoke({
    "problem": "Demodogs in my garden",
    "creativity_level": "high"
}))

Use colored lights to guide the danger away from: Demodogs in my garden.
Ask the party to create a safe escape route for: Demodogs in my garden.
Build a magical fence to protect the garden from: Demodogs in my garden.


### 2d. `gather_party_wisdom` [YOUR TURN]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

In [8]:
# YOUR TURN: party wisdom tool
@tool
def gather_party_wisdom(question: str) -> str:
    """Return advice from a party member named in the question."""

    wisdom = {
        "mike": "Mike says: Stay together and make a simple plan.",
        "dustin": "Dustin says: Check the evidence before guessing.",
        "lucas": "Lucas says: Be careful and keep an escape route.",
        "will": "Will says: Pay attention to strange changes around you.",
    }

    question_lower = question.lower()

    for member, advice in wisdom.items():
        if member in question_lower:
            return advice

    return "The party huddles together and tries to make a plan."

In [15]:
# YOUR TURN: test each tool on its own with .invoke({...})

print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

The Demogorgon ignores 'My lights keep flickering' and walks back into the wall. Classic.
Hawkins reports say portals sometimes appear near the old laboratory.
Use colored lights to guide the danger away from: Demodogs in my garden.
Ask the party to create a safe escape route for: Demodogs in my garden.
Place a harmless decoy far away to distract attention from: Demodogs in my garden.
The party huddles together and tries to make a plan.


In [10]:
print(gather_party_wisdom.invoke({"question": "What does Dustin suggest?"}))

Dustin says: Check the evidence before guessing.


**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent [YOUR TURN]

System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

In [16]:
tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom,
]

print([tool.name for tool in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


In [17]:
# YOUR TURN: tool list + system prompt + agent

system_prompt = """
You are the NormalObjects complaint handler.
For every complaint, use at least one relevant tool before answering.
Base factual claims only on tool outputs. Do not invent facts.
Be creative only when suggesting ideas returned by a tool.
At the end, list the names of the tools you used.
If no matching record is found, say so clearly.
"""

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
)

print("Agent created with tools:", [tool.name for tool in tools])

Agent created with tools: ['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints [YOUR TURN]

Use `config={"recursion_limit": 10}` to stop loops.

In [18]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

# YOUR TURN: run every complaint, keep the results for Step 5

results = []

for complaint in complaints:
    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )

    results.append(result)
    print("\nComplaint:", complaint)
    print("Answer:", result["messages"][-1].content)


Complaint: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
Answer: It seems that there are no records related to your complaint about flickering Christmas lights and noises from the wall. Additionally, when I consulted the Demogorgon about it, it simply ignored the issue and returned to the wall, which is rather unhelpful.

Given the situation, it might be beneficial to check your electrical connections and wiring, as flickering lights can often indicate a loose connection or an issue with the electrical system. If the noises persist, I recommend consulting a qualified electrician to ensure everything is safe.

Tools used: check_hawkins_records, consult_demogorgon.

Complaint: There is a weird portal in my basement. What's the plan?
Answer: It appears that according to Hawkins records, portals sometimes emerge near the old laboratory, which might explain the one in your basement. However, the Demogorgon’s chaotic response suggests that it finds t

In [25]:
for i, result in enumerate(results, start=1):
    print(f"\n--- Complaint {i} ---")

    for msg in result["messages"]:
        if getattr(msg, "type", "") == "human":
            print("Complaint:", msg.content)

        for call in getattr(msg, "tool_calls", []):
            print("Tool call:", call["name"], call["args"])

        if getattr(msg, "name", None):
            print("Tool output:", msg.name)
            print(msg.content)

        if (
            getattr(msg, "type", "") == "ai"
            and msg.content
            and not getattr(msg, "tool_calls", None)
        ):
            print("Agent answer:", msg.content)


--- Complaint 1 ---
Complaint: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
Tool call: check_hawkins_records {'query': 'Christmas lights flickering noises from wall'}
Tool call: consult_demogorgon {'complaint': 'Christmas lights flickering and noises from the wall'}
Tool output: check_hawkins_records
No record found for: Christmas lights flickering noises from wall
Tool output: consult_demogorgon
The Demogorgon ignores 'Christmas lights flickering and noises from the wall' and walks back into the wall. Classic.
Agent answer: It seems that there are no records related to your complaint about flickering Christmas lights and noises from the wall. Additionally, when I consulted the Demogorgon about it, it simply ignored the issue and returned to the wall, which is rather unhelpful.

Given the situation, it might be beneficial to check your electrical connections and wiring, as flickering lights can often indicate a loose connection or an issue wit

In [19]:
for msg in results[2]["messages"]:
    if getattr(msg, "tool_calls", None):
        for call in msg.tool_calls:
            print("Tool call:", call["name"], call["args"])
    if msg.type == "tool":
        print("Tool output:", msg.content)

print("Final answer:", results[2]["messages"][-1].content)

Tool call: cast_interdimensional_spell {'problem': 'A Demodog ate my homework', 'creativity_level': 'high'}
Tool output: Place a harmless decoy far away to distract attention from: A Demodog ate my homework.
Use colored lights to guide the danger away from: A Demodog ate my homework.
Ask the party to create a safe escape route for: A Demodog ate my homework.
Final answer: To creatively solve the issue of your homework being eaten by a Demodog, consider the following high-level spell suggestions:

1. **Create a Harmless Decoy**: Place a harmless decoy far away to distract the Demodog's attention from your homework.
   
2. **Use Colored Lights**: Set up colored lights to guide the Demodog away from your homework and into another area.

3. **Safe Escape Route**: Enlist your party members to help create a safe escape route for you, ensuring that you can retrieve or redo your homework without further interference from the Demodog.

These ideas should help you manage the situation creatively

**Check:** for the cat complaint, does the agent admit "no record", or invent one?

---
## Step 5: Analyze Agent Behavior [YOUR TURN]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

In [20]:
# YOUR TURN: tool order + counts

from collections import Counter

tool_counts = Counter()

for i, result in enumerate(results, start=1):
    order = []

    for msg in result["messages"]:
        if getattr(msg, "tool_calls", None):
            for call in msg.tool_calls:
                name = call["name"]
                order.append(name)
                tool_counts[name] += 1

    print(f"Complaint {i}: {' -> '.join(order)}")

print("Total uses:", dict(tool_counts))

Complaint 1: check_hawkins_records -> consult_demogorgon
Complaint 2: check_hawkins_records -> consult_demogorgon
Complaint 3: cast_interdimensional_spell
Complaint 4: consult_demogorgon -> check_hawkins_records
Total uses: {'check_hawkins_records': 3, 'consult_demogorgon': 3, 'cast_interdimensional_spell': 1}


### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

In [26]:
# YOUR TURN: evidence trace for one result

trace = results[2]["messages"]

for msg in trace:
    for call in getattr(msg, "tool_calls", []):
        print("Tool call:", call["name"], call["args"])

    if getattr(msg, "name", None):
        print("Tool output:", msg.name)
        print(msg.content)

print("Final answer:")
print(trace[-1].content)

Tool call: cast_interdimensional_spell {'problem': 'A Demodog ate my homework', 'creativity_level': 'high'}
Tool output: cast_interdimensional_spell
Place a harmless decoy far away to distract attention from: A Demodog ate my homework.
Use colored lights to guide the danger away from: A Demodog ate my homework.
Ask the party to create a safe escape route for: A Demodog ate my homework.
Final answer:
To creatively solve the issue of your homework being eaten by a Demodog, consider the following high-level spell suggestions:

1. **Create a Harmless Decoy**: Place a harmless decoy far away to distract the Demodog's attention from your homework.
   
2. **Use Colored Lights**: Set up colored lights to guide the Demodog away from your homework and into another area.

3. **Safe Escape Route**: Enlist your party members to help create a safe escape route for you, ensuring that you can retrieve or redo your homework without further interference from the Demodog.

These ideas should help you man

**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

In [23]:
# YOUR TURN: fixed-order handler

def structured_handler(complaint: str) -> str:
    records_result = check_hawkins_records.invoke({
        "query": complaint
    })

    party_result = gather_party_wisdom.invoke({
        "question": complaint
    })

    spell_result = cast_interdimensional_spell.invoke({
        "problem": complaint,
        "creativity_level": "low"
    })

    return (
        f"Hawkins records: {records_result}\n"
        f"Party wisdom: {party_result}\n"
        f"Spell: {spell_result}"
    )

In [24]:
print(structured_handler(complaints[1]))

Hawkins records: Hawkins reports say portals sometimes appear near the old laboratory.
Party wisdom: The party huddles together and tries to make a plan.
Spell: Place a harmless decoy far away to distract attention from: There is a weird portal in my basement. What's the plan?.


| | Agent | Structured |
|---|---|---|
| Same tools every run? |No — tools varied by complaint| Yes — the code always calls the same three tools|
| Tool order predictable? | No — order varied across complaints| Yes — order is fixed in the code|
| Can invent facts? | Yes — it may add claims that are not in the tool output.| No new model-generated claims — it uses fixed tool outputs.|
| Easy to trace? |Partly — tool calls are visible, but the final answer may add unsupported claims. |Yes — the code order and tool outputs are visible. |
| Best for which client case? | Open-ended requests that need flexible tool choices.| Repeated tasks that need a fixed, easy-to-check process.|

---
## Extensions [OPTIONAL]

- More tools (`consult_eleven`, `check_government_files`)
- LLM call inside a tool
- Memory with `InMemorySaver` + `thread_id`
- Streamlit / Gradio UI
- Measure response time and tool calls per complaint

In [ ]:
# YOUR TURN (optional)

# TODO

---
## Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Creative Complaint Handler

## 1. What I built
## 2. Query
## 3. Tool evidence (calls + args + outputs)
## 4. Final output
## 5. Trace (which claim comes from which tool output?)
## 6. Failure or limitation (where can it give a plausible but unsupported answer?)
## 7. Agent vs. structured handler
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Agent doesn't use tools | Clearer docstrings; prompt must require tool use |
| Tools in "wrong" order | Expected for agents. Need fixed order? Use LangGraph |
| Agent loops | Lower `recursion_limit`, improve docstrings |
| Generic answers | More specific tool outputs, lower temperature |
| `ImportError` on `create_agent` | `pip install -U langchain` (needs v1+) |